# [LEC] LangChain 프레임워크 깊이 있는 학습

슬라이드와 설명, 퀴즈로 LangChain의 체인 구성, 도구 호출, 구조화된 출력, 메모리를 익힙니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. LCEL과 Runnable (슬라이드) (퀴즈)
  - 1.1 LCEL의 설계 철학 (슬라이드) (퀴즈)
  - 1.2 Runnable 인터페이스 (슬라이드) (퀴즈)
- 2. Tool Calling (슬라이드) (퀴즈)
  - 2.1 Tool Schema와 도구 선택 (슬라이드) (퀴즈)
  - 2.2 응답 파싱과 실행 흐름 (슬라이드) (퀴즈)
- 3. 구조화된 출력과 메모리 (슬라이드) (퀴즈)
  - 3.1 Output Parser와 Pydantic (슬라이드) (퀴즈)
  - 3.2 Buffer, Summary, VectorStoreRetriever Memory (슬라이드) (퀴즈)
- 4. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("1-3", [1, 2])

이번 시간은 LangChain으로 LLM 애플리케이션을 조립하는 방법을 세 부분으로 나눠 봅니다.

1. LCEL과 Runnable: 프롬프트·모델·파서를 하나의 흐름으로 잇는 방법
2. Tool Calling(도구 호출): 모델이 도구와 인자를 고르고, 코드가 그 도구를 실행하는 구조
3. 구조화된 출력과 메모리: 답을 정해진 모양으로 받고, 이전 대화를 다음 요청에 반영하는 방법

**왜 배우나:** 뒤에서 만드는 RAG 체인과 LangGraph Agent도 이 조각들 위에 쌓입니다. 조각마다 역할을 알아 두면 AI가 만든 체인 코드를 읽고 검토할 수 있습니다.

<details><summary>강사용 메모</summary>

| 항목 | 내용 |
|---|---|
| 원본 PDF | `docs/raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf` (14쪽) |
| 원본 코드 | 이 노트북에는 없습니다. `src/raw/Chapter01/02-langchain/`의 다섯 파일은 `1-3_langchain_LAB.ipynb`에서 다룹니다. |
| API 호출 | 없습니다. 슬라이드 준비 셀만 실행하면 됩니다. |
| 시간 | 커리큘럼 1.3은 2.0H. 목차 초안 배분은 LEC 40분 + LAB 70분 + 정리 10분 |

---

- 실습 코드(`01_lcel_runnable.py`~`05_memory_langgraph.py`)는 LAB 노트북에 있습니다. 이 노트북은 코드 셀 없이 슬라이드·설명·퀴즈만 담습니다.
- 커리큘럼 1.3의 "Output Parser vs with_structured_output() 비교"와 "메모리 전략 2: LangGraph + MemorySaver"는 PDF에 없습니다. LAB 4장·6장에서 예제로만 다룹니다.

</details>

## 1. LCEL과 Runnable <small>(슬라이드) (퀴즈)</small>

### 1.1 LCEL의 설계 철학 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-3", [3, 4])

LCEL(LangChain Expression Language)은 LLM 애플리케이션을 작은 실행 단위의 조합으로 표현하는 LangChain의 체인 구성 방식입니다.
프롬프트, 모델, 파서(모델의 출력을 원하는 모양으로 바꾸는 단위), 함수 등을 하나의 흐름으로 연결합니다.
`|` 연산자는 앞 단계의 출력을 다음 단계의 입력으로 넘깁니다.
슬라이드 코드의 `chain = prompt | model | parser`는 "프롬프트를 채우고, 모델에 보내고, 답을 문자열로 꺼낸다"는 순서를 한 줄로 적은 것입니다.

**왜 배우나:** 복잡한 Agent도 결국 "입력을 변환해 다음 실행 단위로 넘기는 흐름"으로 읽을 수 있습니다. 코드가 길어져도 `|`로 이어진 순서만 따라가면 데이터가 어디로 가는지 알 수 있습니다.

In [ ]:
show_slides("1-3", [5])

LCEL의 핵심 장점은 실행 흐름을 선언적으로(무엇을 어떤 순서로 할지 적는 방식으로) 읽을 수 있다는 점입니다.

- 코드가 데이터 처리 파이프라인처럼 보입니다.
- 각 단계의 역할과 연결 관계가 분명해집니다.
- 동기, 비동기, 스트리밍, 배치 실행 같은 실행 방식을 같은 구조에서 다루기 쉽습니다.

In [ ]:
show_quiz("../assets/quizzes/1-3/q11.json")

**정리:** LCEL은 `|`로 실행 단위를 이어, 앞 단계의 출력을 다음 단계의 입력으로 넘깁니다.

<details><summary>강사용 메모</summary>

- 슬라이드 코드는 `ChatOpenAI(model="gpt-4.1-mini")`로 모델을 만들지만, 실습 코드는 `get_llm()` 래퍼로 모델을 준비합니다. 물으면 "모델을 준비하는 방법만 다르고 체인 구성은 같다"고 답합니다.

</details>

### 1.2 Runnable 인터페이스 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-3", [6])

Runnable은 LangChain에서 실행할 수 있는 구성 요소를 같은 방식으로 다루기 위한 인터페이스(공통 사용법)입니다.
프롬프트 템플릿, LLM, 출력 파서, 직접 만든 함수가 모두 Runnable처럼 연결됩니다.
그래서 같은 체인을 여러 방식으로 실행할 수 있습니다.

- `invoke`: 입력 하나에 결과 하나를 돌려줍니다.
- `batch`: 여러 입력을 목록으로 받아 한 번에 처리합니다.
- `stream`: 답을 조각으로 받아 생기는 대로 출력합니다.
- `ainvoke`: `invoke`의 비동기 버전입니다.

**왜 배우나:** 모델을 호출할 때 쓰던 `invoke()`·`stream()`을 체인 전체에도 똑같이 씁니다. 체인을 한 번 만들어 두면 한 번 묻기, 여러 번 묻기, 조각으로 받기를 호출 한 줄만 바꿔 고를 수 있습니다.

In [ ]:
show_slides("1-3", [7])

Runnable을 조합하면 복잡한 입력 변환을 단계별로 나눌 수 있습니다.

1. `RunnablePassthrough`는 입력을 그대로 다음 단계로 넘깁니다.
2. `RunnableLambda`는 일반 함수를 체인 안의 실행 단위로 감쌉니다.
3. `RunnableParallel`은 여러 실행 결과를 병렬로 만들어 하나의 입력 묶음으로 합칩니다.

**왜 배우나:** 검색 결과, 사용자 질문, 메타데이터(부가 정보)를 하나의 프롬프트 입력으로 모을 때 이 조합을 씁니다. 2일차 RAG 체인에서 "검색한 문서"와 "질문"을 함께 프롬프트에 넣는 부분이 이런 모양입니다.

In [ ]:
show_quiz("../assets/quizzes/1-3/q12.json")

**정리:** 체인의 조각은 모두 Runnable이라 `invoke`·`batch`·`stream`으로 똑같이 실행하고, Passthrough·Lambda·Parallel로 입력을 넘기고 감싸고 모읍니다.

<details><summary>강사용 메모</summary>

- 슬라이드 순서(Passthrough → Lambda → Parallel)와 실습 코드 순서(Parallel → Passthrough → Lambda)가 다릅니다. LAB은 원본 코드 순서를 따릅니다.
- 슬라이드는 "Passthrough는 입력을 그대로 전달"이라고만 적었지만, 실습 코드는 `RunnablePassthrough.assign(length=…)`으로 입력을 넘기면서 새 값을 덧붙입니다. 물으면 "그대로 넘기면서 값을 덧붙일 수도 있다"고 답합니다.
- RAG 연결 근거: `src/raw/Chapter01/03-rag/04_langchain_rag.py`의 `{'context': retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()`.

</details>

## 2. Tool Calling <small>(슬라이드) (퀴즈)</small>

### 2.1 Tool Schema와 도구 선택 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-3", [8, 9])

Tool Calling(도구 호출)은 LLM이 글로 답하는 대신, 쓸 도구와 인자(도구에 넘길 값)를 구조화된 형태로 고르게 하는 방식입니다.
Tool Schema(도구 명세)는 도구 이름, 설명, 입력 인자, 인자 타입을 모델에게 알려 줍니다.
모델은 사용자의 요청과 도구 설명을 비교해 어떤 도구가 필요한지 판단합니다.
그래서 좋은 도구 설명은 도구가 하는 일뿐 아니라 **언제 써야 하는지**를 분명히 말해야 합니다.

- 도구 이름은 짧고 목적이 드러나게 짓습니다.
- 인자 설명은 모델이 값을 잘 채울 수 있을 만큼 구체적으로 씁니다.
- 비슷한 도구가 많으면 고르는 기준을 더합니다.

**왜 배우나:** 모델은 도구의 코드를 보지 않고 이름과 설명, 인자 명세를 보고 고릅니다. 모델이 도구를 엉뚱하게 고른다면 먼저 도구 설명을 살펴봅니다.

In [ ]:
show_quiz("../assets/quizzes/1-3/q21.json")

**정리:** 모델은 도구 이름·설명·인자 명세를 보고 도구를 고르므로, 설명에는 "언제 쓰는지"가 들어가야 합니다.

<details><summary>강사용 메모</summary>

- LAB 3장에서는 `@tool` 함수의 docstring이 도구 설명, 타입 힌트가 인자 타입이 됩니다(원본 주석 "docstring이 LLM에게 전달되는 도구 설명이 된다").

</details>

### 2.2 응답 파싱과 실행 흐름 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-3", [10])

Tool Calling에서 모델은 도구를 직접 실행하지 않습니다. 모델이 밝힌 도구 호출 의도를 바깥의 런타임(코드를 실제로 돌리는 쪽)이 실행합니다.

1. 모델은 호출할 도구 이름과 인자 값을 만듭니다. 슬라이드의 JSON이 그 모양입니다.
2. 애플리케이션이 그 결과를 파싱(읽어서 필요한 값으로 나누기)해 실제 함수를 실행합니다.
3. 함수 실행 결과는 다시 모델 입력으로 들어가 최종 답변을 만드는 데 쓰입니다.

In [ ]:
show_slides("1-3", [11])

Agent의 기본 실행 흐름은 Tool Calling을 되풀이해 씁니다.

1. 사용자의 요청을 보고 필요한 행동을 판단합니다.
2. 알맞은 도구와 입력 인자를 고릅니다.
3. 도구 실행 결과를 관찰합니다.
4. 더 할 일이 있으면 다시 도구를 고르고, 충분하면 최종 답변을 만듭니다.

**왜 배우나:** Agent가 "스스로 판단해 도구를 쓴다"는 것은 이 반복을 말합니다. 이 흐름을 잡아 두면 2일차 LangGraph에서 같은 반복을 그래프로 만들 때도 같은 구조로 읽을 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/1-3/q22.json")

**정리:** 모델은 도구와 인자를 고르고, 실행은 애플리케이션이 합니다. 그 결과가 다시 모델에 들어가는 반복이 Agent의 기본 흐름입니다.

<details><summary>강사용 메모</summary>

- 슬라이드 JSON은 `tool`/`arguments` 이름을 쓰지만, LangChain의 `response.tool_calls` 항목은 `name`/`args`/`id`입니다. LAB 3장은 `print(call)`로 이 항목을 그대로 출력하므로 대응을 화면으로 보여 줄 수 있습니다.

---

- 11차수 진행 원칙대로 "도구 등록 방법"보다 "모델이 어떻게 고르고 호출이 어떻게 반복되는지"를 구조로 설명합니다.
- LAB 3장의 `run_with_tools()`가 이 반복을 `while True` 루프로 직접 짠 것입니다. 2일차 LangGraph 실습의 `ToolNode`(`src/raw/Chapter01/04-langgraph/03_tool_node.py`)가 이 루프를 자동화합니다(원본 주석 근거).

</details>

## 3. 구조화된 출력과 메모리 <small>(슬라이드) (퀴즈)</small>

### 3.1 Output Parser와 Pydantic <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-3", [12, 13])

Output Parser는 LLM의 텍스트 출력을 애플리케이션이 쓸 수 있는 구조화된 데이터로 바꿉니다.

- 일반 텍스트 답변은 사람이 읽기에는 좋지만 프로그램이 안정적으로 처리하기 어렵습니다.
- JSON, 리스트, 객체 모양으로 출력 규칙을 정하면 뒤따르는 처리 코드가 단순해집니다.
- Parser는 모델 출력이 약속한 형식에 맞는지 확인하고 필요한 데이터만 꺼냅니다.

Pydantic(파이썬 데이터 검증 도구)은 구조화된 출력의 스키마(데이터 모양)와 검증 규칙을 코드로 적을 때 자주 씁니다.

- 필드 이름, 타입, 설명을 모델에게 분명히 전할 수 있습니다.
- 잘못된 타입이나 빠진 필드를 검증할 수 있습니다.

**왜 배우나:** 평가, 라우팅(다음에 갈 경로 고르기), 저장, 화면 표시처럼 정해진 데이터 구조가 필요한 작업에 씁니다.

In [ ]:
show_quiz("../assets/quizzes/1-3/q31.json")

**정리:** Output Parser는 텍스트 답을 프로그램이 쓸 데이터로 바꾸고, Pydantic은 그 데이터의 모양과 검증 규칙을 정합니다.

<details><summary>강사용 메모</summary>

- LAB 4장의 `MovieReview`(title·genre·rating·summary·recommend)가 이 스키마의 예입니다.
- LAB 4장의 방법 2 `with_structured_output()`은 PDF에 없어 예제로만 실행합니다(2026-09-28 사용자 결정).

</details>

### 3.2 Buffer, Summary, VectorStoreRetriever Memory <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("1-3", [14])

Memory는 대화나 작업 과정의 이전 정보를 다음 요청에 반영하기 위한 설계입니다.

- Buffer Memory는 이전 대화를 그대로 보관해 모델에게 전달합니다.
- Summary Memory는 긴 대화를 요약해 컨텍스트 사용량을 줄입니다.
- VectorStoreRetriever Memory는 과거 기록을 임베딩(의미를 숫자 묶음으로 바꾸기)해 관련 높은 기억만 검색합니다.

메모리 전략은 정확도, 비용, 지연시간의 균형으로 고릅니다.

- 짧은 대화는 Buffer Memory가 단순하고 정확합니다.
- 긴 대화는 Summary Memory로 핵심 맥락을 압축합니다.
- 누적 지식이 많은 서비스는 VectorStoreRetriever Memory로 필요한 기억만 가져옵니다.

중요한 상태는 자연어 기억보다 별도의 데이터 구조로 저장하는 편이 안정적입니다.

**왜 배우나:** 모델은 이전 대화를 기억하지 않습니다. 대화를 이어 가려면 코드가 이전 대화를 다시 넣어 줘야 하고, 다시 넣는 양이 많을수록 입력 토큰과 비용이 커집니다. 세 메모리는 "무엇을 얼마나 다시 넣을지"를 정하는 서로 다른 방법입니다.

In [ ]:
show_quiz("../assets/quizzes/1-3/q32.json")

**정리:** Buffer는 그대로, Summary는 요약해서, VectorStoreRetriever는 관련 기억만 골라 다시 넣습니다. 정확도·비용·지연시간을 보고 고릅니다.

<details><summary>강사용 메모</summary>

- LAB 코드는 Buffer 방식만 구현합니다(`RunnableWithMessageHistory` 세션 메모리, LangGraph `MemorySaver`). Summary·VectorStoreRetriever 실습 코드는 원본에 없습니다.
- "왜 배우나"는 1-2의 Context Window(모델은 요청에 담긴 것만 본다)와 연결한 설명입니다.

</details>

## 4. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/1-3/final.json")

**이 노트북에서 본 것**
- LCEL은 `|`로 프롬프트·모델·파서·함수를 이어, 앞 단계의 출력을 다음 단계의 입력으로 넘깁니다.
- 체인의 조각은 모두 Runnable이라 `invoke`·`batch`·`stream`으로 똑같이 실행합니다.
- `RunnablePassthrough`는 입력을 넘기고, `RunnableLambda`는 함수를 감싸고, `RunnableParallel`은 여러 결과를 모읍니다.
- Tool Calling에서 모델은 도구와 인자를 고르고, 실행은 애플리케이션이 합니다. 결과가 다시 모델에 들어가는 반복이 Agent의 기본 흐름입니다.
- Output Parser와 Pydantic은 텍스트 답을 프로그램이 쓸 데이터로 바꾸고 검증합니다.
- 메모리는 이전 정보를 다시 넣는 방법이며, Buffer·Summary·VectorStoreRetriever를 정확도·비용·지연시간으로 고릅니다.